# Monte Carlo Strategy Simulation — Block Bootstrap from Empirical SOL/USDC Returns

**Method**: block bootstrap sampling of 7-day price blocks from 1 year of historical data.
Each block preserves within-week serial correlation, fat tails, and volatility clustering.

**Strategies compared**: Bond, Plain LP, Corridor (fixed & two-part), Corridor+Jito, RT Pool,
Put Spread (Bybit real prices), Delta-neutral Perp hedge.

**Configurations**: 3 markups x 2 widths x 5 yield regimes = 30 scenarios.

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import requests, time, struct, base64, json, warnings
from scipy.special import roots_hermite
from scipy import stats
import pandas as pd
from collections import defaultdict

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (12, 6),
                     'font.size': 10, 'axes.titlesize': 12})

# ── Constants ──
N_LIQUIDITY   = 10_000
T_WEEK        = 7 / 365
ALPHA         = 0.40
BOND_APY      = 0.12
JITO_APY      = 0.07
SOL_FRACTION  = 0.48
PERP_FUNDING_APY    = 0.80
PERP_FEE_BPS        = 8
PERP_PRICE_IMPACT_BPS = 3
IV_PREMIUM           = 0.25
OPTION_SPREAD_PCT    = 0.10
REAL_OPTION_COST     = {'5pct': 1.79, '10pct': 2.27}  # Bybit verified $/SOL

WIDTHS = {
    '5pct':  {'w': 0.05, 'daily_fee': 0.0065, 'fs_bps': 2500, 'opt_cost': 1.79},
    '10pct': {'w': 0.10, 'daily_fee': 0.0045, 'fs_bps': 2000, 'opt_cost': 2.27},
}
MARKUPS       = [1.05, 1.10, 1.15]

# ── IV/RV Adaptive Markup ──
MARKUPS_FIXED = MARKUPS  # keep original for backward compat
MARKUP_FLOORS = [1.03, 1.05, 1.07]
DEFAULT_MARKUP_FLOOR = 1.05

IV_RV_PARAMS = {
    'constant_1.25': {'type': 'constant', 'value': 1.25},
    'regime_switching': {
        'type': 'regime',
        'high_vol_iv_rv': 1.40,
        'low_vol_iv_rv': 1.15,
        'threshold': 1.3,
    },
    'empirical': {
        'type': 'lognormal',
        'mu': 0.182,    # log(1.20) ≈ 0.182 → median IV/RV = 1.20
        'sigma': 0.25,
    },
}
# NOTE: In production, IV is fetched from BOTH Binance and Bybit.
# The LOWER ATM IV is used to compute IV/RV, ensuring competitive pricing.
# Here in MC simulation, we use statistical models instead of live data.
FEE_MULTS     = [0.50, 0.75, 1.00, 1.25, 1.50]
N_PATHS_MAIN  = 5000
N_PATHS_BRK   = 2000
N_WEEKS       = 52

rng = np.random.default_rng(42)
print("Constants loaded. N_PATHS_MAIN={}, N_WEEKS={}".format(N_PATHS_MAIN, N_WEEKS))

Constants loaded. N_PATHS_MAIN=5000, N_WEEKS=52


In [2]:
# ── IV/RV Ratio Generator ──
def generate_iv_rv_ratio(block_vols, ann_vol, params, rng_local):
    n = len(block_vols) if hasattr(block_vols, '__len__') else 1
    if params['type'] == 'constant':
        return np.full(n, params['value'])
    elif params['type'] == 'regime':
        ratio = block_vols / max(ann_vol, 0.01)
        return np.where(ratio > params['threshold'], params['high_vol_iv_rv'], params['low_vol_iv_rv'])
    elif params['type'] == 'lognormal':
        return np.clip(rng_local.lognormal(params['mu'], params['sigma'], size=n), 0.5, 3.0)
    return np.full(n, 1.25)

def adaptive_markup_vec(iv_rv_ratios, floor):
    return np.maximum(floor, iv_rv_ratios)

print('IV/RV generator and adaptive markup functions defined.')

IV/RV generator and adaptive markup functions defined.


In [3]:
# ── Fetch 1 year of daily SOL/USDC from Birdeye (90-day chunks) ──
BIRDEYE_KEY = "ed577a4a6a4f480fa659b4f18673e4b1"
SOL_MINT    = "So11111111111111111111111111111111111111112"

now_ts = int(time.time())
one_year_ago = now_ts - 365 * 86400

all_candles = []
chunk_start = one_year_ago
while chunk_start < now_ts:
    chunk_end = min(chunk_start + 90 * 86400, now_ts)
    url = (f"https://public-api.birdeye.so/defi/ohlcv"
           f"?address={SOL_MINT}&type=1D"
           f"&time_from={chunk_start}&time_to={chunk_end}")
    headers = {"X-API-KEY": BIRDEYE_KEY, "accept": "application/json"}
    resp = requests.get(url, headers=headers, timeout=30)
    data = resp.json()
    items = data.get("data", {}).get("items", [])
    for c in items:
        all_candles.append({'t': c['unixTime'], 'close': c['c']})
    print(f"  chunk {chunk_start}->{chunk_end}: {len(items)} candles")
    chunk_start = chunk_end
    time.sleep(0.5)

# Deduplicate by timestamp, sort
seen = {}
for c in all_candles:
    seen[c['t']] = c['close']
timestamps = sorted(seen.keys())
hist_prices = np.array([seen[t] for t in timestamps], dtype=float)
print(f"\nTotal unique daily closes: {len(hist_prices)}")
print(f"Date range: {pd.Timestamp(timestamps[0], unit='s').date()} to "
      f"{pd.Timestamp(timestamps[-1], unit='s').date()}")
print(f"Price range: ${hist_prices.min():.2f} - ${hist_prices.max():.2f}")

# ── Fetch current price from Orca via Helius RPC ──
HELIUS_RPC = "https://mainnet.helius-rpc.com/?api-key=2ef5fdd0-5c3b-4ae1-a2fc-e12b3fd605e7"
POOL_ACCT  = "Czfq3xZZDmsdGdUyrNLtRhGc47cXcZtLG4crryfu44zE"

resp_rpc = requests.post(HELIUS_RPC, json={
    "jsonrpc": "2.0", "id": 1, "method": "getAccountInfo",
    "params": [POOL_ACCT, {"encoding": "base64"}]
}, timeout=15)
acct_data = base64.b64decode(resp_rpc.json()["result"]["value"]["data"][0])
sqrt_price_x64 = int.from_bytes(acct_data[65:81], 'little')
S0_orca = (sqrt_price_x64 / 2**64)**2 * 1e3
print(f"\nOrca live SOL/USDC price: ${S0_orca:.2f}")

S0 = S0_orca

  chunk 1744453662->1752229662: 90 candles


  chunk 1752229662->1760005662: 90 candles


  chunk 1760005662->1767781662: 90 candles


  chunk 1767781662->1775557662: 90 candles


  chunk 1775557662->1775989662: 5 candles



Total unique daily closes: 365
Date range: 2025-04-13 to 2026-04-12
Price range: $77.88 - $247.54



Orca live SOL/USDC price: $82.30


In [4]:
# ── Return distribution analysis ──
log_returns = np.diff(np.log(hist_prices))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram vs normal
ax = axes[0]
ax.hist(log_returns, bins=50, density=True, alpha=0.7, color='steelblue',
        edgecolor='white', label='Empirical')
x_grid = np.linspace(log_returns.min(), log_returns.max(), 200)
ax.plot(x_grid, stats.norm.pdf(x_grid, log_returns.mean(), log_returns.std()),
        'r-', lw=2, label='Normal fit')
ax.set_title('Daily Log-Return Distribution')
ax.set_xlabel('Log return'); ax.set_ylabel('Density'); ax.legend()

# QQ plot
ax = axes[1]
stats.probplot(log_returns, dist="norm", plot=ax)
ax.set_title('QQ Plot vs Normal')

plt.tight_layout()
plt.savefig('data/mc_return_distribution.png', bbox_inches='tight')
plt.close()

sk = stats.skew(log_returns)
ku = stats.kurtosis(log_returns)
jb_stat, jb_p = stats.jarque_bera(log_returns)
print(f"Daily log-return stats (n={len(log_returns)}):")
print(f"  Mean:     {log_returns.mean():.6f}")
print(f"  Std:      {log_returns.std():.6f}")
print(f"  Skew:     {sk:.4f}")
print(f"  Kurtosis: {ku:.4f}  (normal=0)")
print(f"  JB test:  stat={jb_stat:.2f}, p={jb_p:.6f}")
print(f"  Ann. vol: {log_returns.std()*np.sqrt(365):.1%}")
if ku > 0.5:
    print("  -> Fat tails detected: block bootstrap will preserve this.")

Daily log-return stats (n=364):
  Mean:     -0.001219
  Std:      0.038495
  Skew:     -0.1337
  Kurtosis: 1.6266  (normal=0)
  JB test:  stat=41.21, p=0.000000
  Ann. vol: 73.5%
  -> Fat tails detected: block bootstrap will preserve this.


In [5]:
# ── Build overlapping 8-price blocks (1 week = 7 daily steps) ──
# block[0] = entry price, block[7] = exit price, block[1:7] = daily checks
block_len = 8
weekly_blocks = np.array([hist_prices[i:i+block_len]
                          for i in range(len(hist_prices) - block_len + 1)])
print(f"Weekly blocks shape: {weekly_blocks.shape}")
print(f"  -> {weekly_blocks.shape[0]} overlapping blocks of {block_len} prices each")
print(f"  Entry price range: ${weekly_blocks[:,0].min():.2f} - ${weekly_blocks[:,0].max():.2f}")
print(f"  Weekly return range: {(weekly_blocks[:,-1]/weekly_blocks[:,0]-1).min():.2%} to "
      f"{(weekly_blocks[:,-1]/weekly_blocks[:,0]-1).max():.2%}")

Weekly blocks shape: (358, 8)
  -> 358 overlapping blocks of 8 prices each
  Entry price range: $77.88 - $247.54
  Weekly return range: -33.54% to 25.17%


In [6]:
# ── Block bootstrap sanity check ──
weekly_rets = weekly_blocks[:,-1] / weekly_blocks[:,0] - 1
within_block_vols = np.array([np.std(np.diff(np.log(b))) for b in weekly_blocks])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(weekly_rets, bins=40, alpha=0.7, color='steelblue', edgecolor='white')
axes[0].set_title('Weekly Return Distribution (from blocks)')
axes[0].set_xlabel('Weekly return')
axes[0].axvline(0, color='k', ls='--', alpha=0.5)

axes[1].hist(within_block_vols * np.sqrt(365), bins=30, alpha=0.7, color='coral', edgecolor='white')
axes[1].set_title('Annualized Vol per Block')
axes[1].set_xlabel('Ann. vol')
plt.tight_layout()
plt.savefig('data/mc_block_stats.png', bbox_inches='tight')
plt.close()
print(f"Block weekly returns: mean={weekly_rets.mean():.4%}, std={weekly_rets.std():.4%}")
print(f"Block intra-week vols: mean={within_block_vols.mean():.4f}, "
      f"std={within_block_vols.std():.4f}")

Block weekly returns: mean=-0.4497%, std=9.3144%
Block intra-week vols: mean=0.0335, std=0.0137


## Part II: Core Functions

Concentrated liquidity math, corridor payoff, fair-value quadrature, and BS put pricing.

In [7]:
# ── Core math functions ──

def cl_value_vec(S, L, p_l, p_u):
    """Concentrated liquidity position value (vectorized)."""
    S = np.atleast_1d(np.asarray(S, float))
    spl, spu = np.sqrt(p_l), np.sqrt(p_u)
    sp = np.sqrt(np.clip(S, 1e-10, None))
    bl = S <= p_l
    ab = S >= p_u
    a = np.where(bl, L * (spu - spl) / (spl * spu),
                 np.where(ab, 0., L * (spu - sp) / (sp * spu)))
    b = np.where(bl, 0.,
                 np.where(ab, L * (spu - spl), L * (sp - spl)))
    return a * S + b


def corridor_payoff_vec(S_T, S0, B, Cap, L, p_l, p_u):
    """Corridor certificate payout (vectorized)."""
    S_T = np.atleast_1d(np.asarray(S_T, float))
    V0 = cl_value_vec(S0, L, p_l, p_u)
    V_eff = cl_value_vec(np.maximum(S_T, B), L, p_l, p_u)
    return np.minimum(Cap, np.maximum(0., np.where(S_T >= S0, 0., V0 - V_eff)))


def fv_quadrature(S0, sigma, T=T_WEEK, B=None, Cap=None, L=None, p_l=None, p_u=None):
    """Fair value of corridor certificate via Gauss-Hermite quadrature."""
    nodes, weights = roots_hermite(128)
    S_T = S0 * np.exp(-sigma**2 / 2 * T + sigma * np.sqrt(T) * nodes * np.sqrt(2))
    payoffs = corridor_payoff_vec(S_T, S0, B, Cap, L, p_l, p_u)
    return max(0, np.sum(weights * payoffs) / np.sqrt(np.pi))


def bs_put(S, K, sigma, T, r=0.0):
    """Black-Scholes put price."""
    if T <= 0 or sigma <= 0:
        return max(0., K - S)
    d1 = (np.log(S / K) + (r + sigma**2 / 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return K * np.exp(-r * T) * stats.norm.cdf(-d2) - S * stats.norm.cdf(-d1)


def cl_delta(S, L, p_l, p_u):
    """Numerical delta of CL position."""
    return float((cl_value_vec(S + 0.01, L, p_l, p_u) -
                  cl_value_vec(S - 0.01, L, p_l, p_u)) / 0.02)


def make_position(S, w, L=N_LIQUIDITY):
    """Create position parameters from entry price and width."""
    p_l = S * (1 - w)
    p_u = S * (1 + w)
    V0 = float(cl_value_vec(S, L, p_l, p_u)[0])
    B = p_l  # barrier at lower tick
    Cap = V0 * 0.30  # 30% max protection
    return {'p_l': p_l, 'p_u': p_u, 'V0': V0, 'B': B, 'Cap': Cap, 'L': L}


print("Core functions defined.")

Core functions defined.


In [8]:
# ── Sanity check at current price ──
for wk, wcfg in WIDTHS.items():
    pos = make_position(S0, wcfg['w'])
    delta = cl_delta(S0, N_LIQUIDITY, pos['p_l'], pos['p_u'])
    ann_vol = log_returns.std() * np.sqrt(365)
    fv = fv_quadrature(S0, ann_vol, T=T_WEEK, B=pos['B'], Cap=pos['Cap'],
                       L=N_LIQUIDITY, p_l=pos['p_l'], p_u=pos['p_u'])
    print(f"Width {wk}: V0=${pos['V0']:.2f}, Cap=${pos['Cap']:.2f}, "
          f"B=${pos['B']:.2f}, delta={delta:.4f}")
    print(f"  FV(sigma={ann_vol:.1%})=${fv:.2f}, "
          f"FV/V0={fv/pos['V0']:.4%}, "
          f"FV/Cap={fv/pos['Cap']:.4%}")
    print()

Width 5pct: V0=$4483.26, Cap=$1344.98, B=$78.18, delta=26.5662
  FV(sigma=73.5%)=$69.27, FV/V0=1.5451%, FV/Cap=5.1503%

Width 10pct: V0=$8877.00, Cap=$2663.10, B=$74.07, delta=51.2998
  FV(sigma=73.5%)=$205.22, FV/V0=2.3118%, FV/Cap=7.7060%



## Part III: Monte Carlo Engine — Block Bootstrap

For each simulated week, we sample a historical 8-day block and replay the **actual**
price ratios. This preserves intra-week correlation, fat tails, and vol clustering.

In [9]:
def run_block_bootstrap_mc(markup, width_key, wcfg, weekly_blocks, S0,
                           n_paths=N_PATHS_MAIN, n_weeks=N_WEEKS,
                           daily_fee_override=None,
                           adaptive_mode=False, iv_rv_params=None, markup_floor=DEFAULT_MARKUP_FLOOR):
    """
    Block bootstrap Monte Carlo simulation.

    Returns dict: strategy_name -> final wealth array of shape (n_paths,).
    """
    w = wcfg['w']
    daily_fee = daily_fee_override if daily_fee_override is not None else wcfg['daily_fee']
    fs_bps = wcfg['fs_bps']
    fs_frac = fs_bps / 10_000
    opt_cost_per_sol = wcfg['opt_cost']
    MARKUP = markup

    n_blocks = len(weekly_blocks)
    ann_vol = log_returns.std() * np.sqrt(365)
    sigma_full = ann_vol  # alias for adaptive mode

    # Strategy wealth arrays — multiplicative tracking
    strategies = ['Bond', 'Plain LP', 'Corridor (fixed)', 'Corridor (two-part)',
                  'Corridor+Jito', 'RT Pool', 'Put Spread', 'Perp Hedge']
    wealth = {s: np.ones(n_paths) for s in strategies}

    # Precompute median position for FV scaling
    S_median = np.median(weekly_blocks[:, 0])
    pos_med = make_position(S_median, w)
    fv_med = fv_quadrature(S_median, ann_vol, T=T_WEEK, B=pos_med['B'],
                           Cap=pos_med['Cap'], L=N_LIQUIDITY,
                           p_l=pos_med['p_l'], p_u=pos_med['p_u'])
    V0_median = pos_med['V0']

    # Bond weekly return
    bond_weekly = (1 + BOND_APY) ** (1 / 52) - 1
    # Jito weekly
    jito_weekly = (1 + JITO_APY) ** (1 / 52) - 1

    # Sample all block indices upfront: (n_paths, n_weeks)
    block_indices = rng.integers(0, n_blocks, size=(n_paths, n_weeks))

    for week_idx in range(n_weeks):
        # Get blocks for this week across all paths
        bidx = block_indices[:, week_idx]       # (n_paths,)
        blocks = weekly_blocks[bidx]            # (n_paths, 8)

        S_start = blocks[:, 0]                  # (n_paths,)
        S_end   = blocks[:, 7]                  # (n_paths,)
        daily_prices = blocks[:, 1:7]           # (n_paths, 6)

        # Position parameters per path
        p_l = S_start * (1 - w)
        p_u = S_start * (1 + w)
        V0  = cl_value_vec(S_start, N_LIQUIDITY, p_l, p_u)  # (n_paths,)
        V1  = cl_value_vec(S_end, N_LIQUIDITY, p_l, p_u)    # (n_paths,)

        # IL
        IL = V1 - V0  # typically negative when price moves

        # Fees: simple model = daily_fee * V0 * 7
        fees = daily_fee * V0 * 7

        # In-range check for fee scaling (simplified: use entry/exit)
        # Full: check all 6 daily prices
        in_range_days = np.sum(
            (daily_prices >= p_l[:, None]) & (daily_prices <= p_u[:, None]),
            axis=1)  # out of 6
        # Also check entry and exit
        in_range_entry = ((S_start >= p_l) & (S_start <= p_u)).astype(float)
        in_range_exit  = ((S_end >= p_l) & (S_end <= p_u)).astype(float)
        in_range_frac = (in_range_days + in_range_entry + in_range_exit) / 8.0
        fees = fees * in_range_frac  # scale fees by time in range

        # Corridor payoff
        B_arr = p_l  # barrier = lower tick
        Cap_arr = V0 * 0.30
        payout = corridor_payoff_vec(S_end, S_start, B_arr, Cap_arr,
                                     N_LIQUIDITY, p_l, p_u)

        # Fair value scaled from median
        fv_scaled = fv_med * (V0 / V0_median)

        # Vol indicator from block daily returns
        daily_log_rets = np.diff(np.log(blocks), axis=1)  # (n_paths, 7)
        block_vol = np.std(daily_log_rets, axis=1) * np.sqrt(365)
        vol_ind = np.clip(block_vol / ann_vol, 0.5, 2.0)

        # ── Effective markup ──
        if adaptive_mode and iv_rv_params is not None:
            iv_rv_week = generate_iv_rv_ratio(block_vol, sigma_full, iv_rv_params, rng)
            eff_markup = adaptive_markup_vec(iv_rv_week, markup_floor)
        else:
            eff_markup = MARKUP  # scalar from parameter

        # Weekly fee expected for beta calc
        weekly_fee_exp = V0 * daily_fee * 7

        # ── 1. Bond ──
        wealth['Bond'] *= (1 + bond_weekly)

        # ── 2. Plain LP ──
        ret_lp = (IL + fees) / V0
        wealth['Plain LP'] *= (1 + ret_lp)

        # ── 3. Corridor (fixed markup) ──
        prem_fixed = np.maximum(0, fv_scaled * eff_markup - fees * fs_frac)
        ret_corr_fixed = (IL + fees + payout - prem_fixed) / V0
        wealth['Corridor (fixed)'] *= (1 + ret_corr_fixed)

        # ── 4. Corridor (two-part) ──
        beta = (eff_markup - ALPHA) * fv_scaled / np.maximum(weekly_fee_exp, 1.0)
        prem_twopart = ALPHA * fv_scaled * vol_ind + beta * fees
        ret_corr_tp = (IL + fees + payout - prem_twopart) / V0
        wealth['Corridor (two-part)'] *= (1 + ret_corr_tp)

        # ── 5. Corridor + Jito ──
        jito_return = SOL_FRACTION * jito_weekly
        ret_corr_jito = ret_corr_tp + jito_return
        wealth['Corridor+Jito'] *= (1 + ret_corr_jito)

        # ── 6. RT Pool ──
        nat_cap = Cap_arr
        n_certs = np.maximum(1, np.floor(V0 * 0.30 / nat_cap)).astype(int)
        rt_prem_collected = n_certs * prem_twopart * 0.985  # 1.5% protocol fee
        rt_claims = n_certs * payout
        ret_rt = (rt_prem_collected - rt_claims) / V0
        wealth['RT Pool'] *= (1 + ret_rt)

        # ── 7. Put Spread (Bybit real) ──
        cost_per_sol = opt_cost_per_sol
        n_sol = V0 / S_start
        option_cost = cost_per_sol * n_sol
        barrier_put = S_start * (1 - w)
        # Put spread payoff: long put at S_start, short put at barrier
        put_pay = n_sol * np.maximum(0,
            np.maximum(0, S_start - S_end) - np.maximum(0, barrier_put - S_end))
        ret_put = (IL + fees + put_pay - option_cost) / V0
        wealth['Put Spread'] *= (1 + ret_put)

        # ── 8. Perp (delta-neutral) ──
        delta_vals = np.array([cl_delta(s, N_LIQUIDITY, pl, pu)
                               for s, pl, pu in zip(S_start[:20], p_l[:20], p_u[:20])])
        # Approximate delta for remaining paths using mean ratio
        if len(S_start) > 20:
            delta_ratio = np.mean(delta_vals / S_start[:20])
            delta_all = np.full(n_paths, delta_ratio) * S_start
            delta_all[:20] = delta_vals
        else:
            delta_all = delta_vals
        perp_pnl = -delta_all * (S_end - S_start)
        perp_cost = np.abs(delta_all) * S_start * (
            PERP_FUNDING_APY * T_WEEK +
            2 * PERP_FEE_BPS / 10_000 +
            2 * PERP_PRICE_IMPACT_BPS / 10_000)
        ret_perp = (IL + fees + perp_pnl - perp_cost) / V0
        wealth['Perp Hedge'] *= (1 + ret_perp)

    return wealth


print("MC engine defined.")

MC engine defined.


In [10]:
# ── Run all 30 configurations ──
all_results = {}
total_configs = len(MARKUPS) * len(WIDTHS) * len(FEE_MULTS)
done = 0

for markup in MARKUPS:
    for wk, wcfg in WIDTHS.items():
        for fm in FEE_MULTS:
            daily_fee_ov = wcfg['daily_fee'] * fm
            result = run_block_bootstrap_mc(
                markup, wk, wcfg, weekly_blocks, S0,
                n_paths=N_PATHS_MAIN, n_weeks=N_WEEKS,
                daily_fee_override=daily_fee_ov)
            all_results[(markup, wk, fm)] = result
            done += 1
            if done % 5 == 0 or done == total_configs:
                print(f"  [{done}/{total_configs}] markup={markup}, "
                      f"width={wk}, fee_mult={fm:.2f}")

print(f"\nAll {total_configs} configurations complete.")


# ── Adaptive markup sweep ──
import time as _timer
print('\nRunning adaptive markup configurations...')
for iv_name, iv_params in IV_RV_PARAMS.items():
    for floor in MARKUP_FLOORS:
        for wk, wcfg in WIDTHS.items():
            key = ('adaptive', iv_name, floor, wk, 1.0)
            t0 = _timer.time()
            result = run_block_bootstrap_mc(
                0, wk, wcfg, weekly_blocks, S0,
                n_paths=3000, n_weeks=N_WEEKS,
                adaptive_mode=True, iv_rv_params=iv_params,
                markup_floor=floor)
            all_results[key] = result
            print(f'  {iv_name}/floor={floor}/{wk}: {_timer.time()-t0:.1f}s')

  [5/30] markup=1.05, width=5pct, fee_mult=1.50


  [10/30] markup=1.05, width=10pct, fee_mult=1.50


  [15/30] markup=1.1, width=5pct, fee_mult=1.50


  [20/30] markup=1.1, width=10pct, fee_mult=1.50


  [25/30] markup=1.15, width=5pct, fee_mult=1.50


...

  constant_1.25/floor=1.05/5pct: 0.1s
  constant_1.25/floor=1.05/10pct: 0.1s
  constant_1.25/floor=1.07/5pct: 0.1s
  constant_1.25/floor=1.07/10pct: 0.1s


  regime_switching/floor=1.03/5pct: 0.1s
  regime_switching/floor=1.03/10pct: 0.1s
  regime_switching/floor=1.05/5pct: 0.1s
  regime_switching/floor=1.05/10pct: 0.1s


  regime_switching/floor=1.07/5pct: 0.1s
  regime_switching/floor=1.07/10pct: 0.1s
  empirical/floor=1.03/5pct: 0.1s


  empirical/floor=1.03/10pct: 0.1s
  empirical/floor=1.05/5pct: 0.1s
  empirical/floor=1.05/10pct: 0.1s


  empirical/floor=1.07/5pct: 0.1s
  empirical/floor=1.07/10pct: 0.1s


In [11]:
def compute_stats(wealth_dict, label=''):
    """Compute summary statistics for each strategy."""
    rows = []
    for strat, w in wealth_dict.items():
        ann_ret = w ** (1 / 1) - 1  # already 52 weeks = 1 year
        median_ann = np.median(ann_ret)
        mean_ann = np.mean(ann_ret)
        std_ann = np.std(ann_ret)
        sharpe = mean_ann / std_ann if std_ann > 0 else 0
        p_positive = np.mean(ann_ret > 0)
        pct5 = np.percentile(ann_ret, 5)
        bond_total = (1 + BOND_APY) - 1
        p_beat_bond = np.mean(ann_ret > bond_total)
        rows.append({
            'Strategy': strat,
            'Median Ann': median_ann,
            'Mean Ann': mean_ann,
            'Std Ann': std_ann,
            'Sharpe': sharpe,
            'P(+)': p_positive,
            '5th%': pct5,
            'P(>Bond)': p_beat_bond,
        })
    df = pd.DataFrame(rows)
    return df


def print_stats_table(df, title=''):
    """Pretty print a stats table."""
    if title:
        print(f"\n{'='*70}")
        print(f"  {title}")
        print(f"{'='*70}")
    fmt = df.copy()
    for col in ['Median Ann', 'Mean Ann', 'Std Ann', '5th%']:
        fmt[col] = fmt[col].map(lambda x: f"{x:+.2%}")
    for col in ['Sharpe']:
        fmt[col] = fmt[col].map(lambda x: f"{x:.3f}")
    for col in ['P(+)', 'P(>Bond)']:
        fmt[col] = fmt[col].map(lambda x: f"{x:.1%}")
    print(fmt.to_string(index=False))


print("Stats helpers defined.")

Stats helpers defined.


In [12]:
# ── Quick preview: markup=1.10, both widths, reference fees ──
for wk in WIDTHS:
    res = all_results[(1.10, wk, 1.00)]
    df = compute_stats(res)
    print_stats_table(df, f"Preview: markup=1.10, width={wk}, fee_mult=1.00")
    print()


  Preview: markup=1.10, width=5pct, fee_mult=1.00
           Strategy Median Ann Mean Ann Std Ann Sharpe   P(+)    5th% P(>Bond)
               Bond    +12.00%  +12.00%  +0.00%  0.000 100.0% +12.00%   100.0%
           Plain LP     -8.85%   -0.17% +48.06% -0.004  42.2% -61.76%    33.9%
   Corridor (fixed)    +19.57%  +26.17% +49.22%  0.532  66.5% -42.35%    56.3%
Corridor (two-part)     +7.03%  +11.95% +41.68%  0.287  56.8% -46.78%    44.8%
      Corridor+Jito    +10.57%  +15.64% +43.03%  0.363  59.5% -45.00%    48.5%
            RT Pool    -12.43%  -11.82% +11.09% -1.066  14.7% -28.92%     2.1%
         Put Spread    +38.45%  +43.73% +49.87%  0.877  80.4% -28.19%    71.5%
         Perp Hedge    -30.85%  -27.67% +24.52% -1.129  13.2% -61.68%     6.3%


  Preview: markup=1.10, width=10pct, fee_mult=1.00
           Strategy Median Ann Mean Ann Std Ann Sharpe   P(+)    5th% P(>Bond)
               Bond    +12.00%  +12.00%  +0.00%  0.000 100.0% +12.00%   100.0%
           Plain LP    +26.

In [13]:
# ── Adaptive vs Fixed: Comparison ──
print('=' * 100)
print('IV/RV-ADAPTIVE vs FIXED MARKUP')
print('=' * 100)

for wk in ['5pct', '10pct']:
    print(f'\n--- {wk} ---')
    print(f'{"Config":<45} {"Corr Med":>10} {"Corr Shrp":>10} {"RT Med":>10} {"RT Mean":>10} {"Viable":>8}')
    print('-' * 95)

    # Fixed baselines
    for m in MARKUPS_FIXED:
        key = (m, wk, 1.0)
        if key not in all_results:
            continue
        w = all_results[key]
        c_med = (np.median(w['Corridor (two-part)']) - 1) * 100
        c_sh = np.mean(w['Corridor (two-part)'] - 1) / max(np.std(w['Corridor (two-part)'] - 1), 1e-10)
        r_med = (np.median(w['RT Pool']) - 1) * 100
        r_mn = (np.mean(w['RT Pool']) - 1) * 100
        viable = 'YES' if c_med > 0 and r_mn > 0 else 'NO'
        print(f'{"Fixed " + str(m) + "x":<45} {c_med:>+9.1f}% {c_sh:>9.3f} {r_med:>+9.1f}% {r_mn:>+9.1f}% {viable:>8}')

    # Adaptive
    for iv_name in IV_RV_PARAMS:
        for floor in MARKUP_FLOORS:
            key = ('adaptive', iv_name, floor, wk, 1.0)
            if key not in all_results:
                continue
            w = all_results[key]
            c_med = (np.median(w['Corridor (two-part)']) - 1) * 100
            c_sh = np.mean(w['Corridor (two-part)'] - 1) / max(np.std(w['Corridor (two-part)'] - 1), 1e-10)
            r_med = (np.median(w['RT Pool']) - 1) * 100
            r_mn = (np.mean(w['RT Pool']) - 1) * 100
            viable = 'YES' if c_med > 0 and r_mn > 0 else 'NO'
            label = f'Adaptive {iv_name}/floor={floor}'
            print(f'{label:<45} {c_med:>+9.1f}% {c_sh:>9.3f} {r_med:>+9.1f}% {r_mn:>+9.1f}% {viable:>8}')

IV/RV-ADAPTIVE vs FIXED MARKUP

--- 5pct ---
Config                                          Corr Med  Corr Shrp     RT Med    RT Mean   Viable
-----------------------------------------------------------------------------------------------
Fixed 1.05x                                        +8.4%     0.341     -14.9%     -14.2%       NO
Fixed 1.1x                                         +7.0%     0.287     -12.4%     -11.8%       NO
Fixed 1.15x                                        +5.7%     0.244     -10.3%      -9.9%       NO
Adaptive constant_1.25/floor=1.03                  +0.2%     0.102      -5.4%      -5.1%       NO
Adaptive constant_1.25/floor=1.05                  -1.6%     0.073      -6.3%      -5.4%       NO
Adaptive constant_1.25/floor=1.07                  -2.0%     0.059      -6.2%      -5.4%       NO
Adaptive regime_switching/floor=1.03               +3.5%     0.190     -10.0%      -9.2%       NO
Adaptive regime_switching/floor=1.05               +3.2%     0.198      -9

## Part IV: Results at Reference Yield (fee_mult=1.0)

Detailed statistics and comparison charts for all markup x width combinations
at the reference fee rate.

In [14]:
# ── Full tables at reference yield ──
for markup in MARKUPS:
    for wk in WIDTHS:
        res = all_results[(markup, wk, 1.00)]
        df = compute_stats(res)
        print_stats_table(df, f"Markup={markup}, Width={wk}, Fees=reference")

        # LP/RT viability check
        lp_med = df.loc[df['Strategy'] == 'Corridor (two-part)', 'Median Ann'].values[0]
        rt_med = df.loc[df['Strategy'] == 'RT Pool', 'Median Ann'].values[0]
        lp_ok = lp_med > 0
        rt_ok = rt_med > 0
        status = "BOTH VIABLE" if (lp_ok and rt_ok) else (
            "LP only" if lp_ok else ("RT only" if rt_ok else "NEITHER"))
        print(f"  -> Viability: {status} (LP median={lp_med:+.2%}, RT median={rt_med:+.2%})")
        print()


  Markup=1.05, Width=5pct, Fees=reference
           Strategy Median Ann Mean Ann Std Ann Sharpe   P(+)    5th% P(>Bond)
               Bond    +12.00%  +12.00%  +0.00%  0.000 100.0% +12.00%   100.0%
           Plain LP    -10.09%   -0.97% +47.14% -0.021  41.5% -60.73%    32.5%
   Corridor (fixed)    +22.85%  +30.72% +50.05%  0.614  70.6% -38.85%    60.2%
Corridor (two-part)     +8.37%  +14.27% +41.90%  0.341  58.7% -45.01%    46.9%
      Corridor+Jito    +11.95%  +18.03% +43.25%  0.417  62.1% -43.17%    49.9%
            RT Pool    -14.87%  -14.23% +10.63% -1.339   9.4% -30.37%     1.4%
         Put Spread    +37.38%  +43.36% +48.88%  0.887  81.1% -27.09%    72.0%
         Perp Hedge    -30.17%  -27.27% +23.66% -1.152  12.4% -60.46%     6.4%
  -> Viability: LP only (LP median=+8.37%, RT median=-14.87%)


  Markup=1.05, Width=10pct, Fees=reference
           Strategy Median Ann Mean Ann Std Ann Sharpe   P(+)    5th% P(>Bond)
               Bond    +12.00%  +12.00%  +0.00%  0.000 100.0

In [15]:
# ── Bar chart: 3x2 grid (markups x widths) ──
fig, axes = plt.subplots(len(MARKUPS), len(WIDTHS), figsize=(16, 14), sharey=True)
strategies_order = ['Bond', 'Plain LP', 'Corridor (fixed)', 'Corridor (two-part)',
                    'Corridor+Jito', 'RT Pool', 'Put Spread', 'Perp Hedge']
colors = plt.cm.Set2(np.linspace(0, 1, len(strategies_order)))

for i, markup in enumerate(MARKUPS):
    for j, wk in enumerate(WIDTHS):
        ax = axes[i, j] if len(MARKUPS) > 1 else axes[j]
        res = all_results[(markup, wk, 1.00)]
        medians = [np.median(res[s] - 1) for s in strategies_order]
        bars = ax.bar(range(len(strategies_order)), medians, color=colors,
                      edgecolor='gray', alpha=0.85)
        ax.set_title(f"Markup={markup}, {wk}")
        ax.set_xticks(range(len(strategies_order)))
        ax.set_xticklabels([s.replace(' ', '\n') for s in strategies_order],
                           rotation=45, ha='right', fontsize=7)
        ax.axhline(0, color='k', ls='--', lw=0.8)
        ax.axhline(BOND_APY, color='green', ls=':', lw=0.8, alpha=0.5)
        ax.set_ylabel('Median Annual Return' if j == 0 else '')
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))

plt.suptitle('Strategy Comparison at Reference Yield (fee_mult=1.0)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('data/mc_strategy_comparison_all.png', bbox_inches='tight')
plt.close()
print("Saved: data/mc_strategy_comparison_all.png")

Saved: data/mc_strategy_comparison_all.png


In [16]:
# ── Markup sensitivity: Corridor and RT metrics across markups ──
for wk in WIDTHS:
    print(f"\n{'='*70}")
    print(f"  Markup Sensitivity — Width={wk}")
    print(f"{'='*70}")
    rows = []
    for markup in MARKUPS:
        res = all_results[(markup, wk, 1.00)]
        df = compute_stats(res)
        corr = df[df['Strategy'] == 'Corridor (two-part)'].iloc[0]
        rt = df[df['Strategy'] == 'RT Pool'].iloc[0]
        rows.append({
            'Markup': markup,
            'Corr Median': f"{corr['Median Ann']:+.2%}",
            'Corr Sharpe': f"{corr['Sharpe']:.3f}",
            'Corr P(+)': f"{corr['P(+)']:.1%}",
            'RT Median': f"{rt['Median Ann']:+.2%}",
            'RT Mean': f"{rt['Mean Ann']:+.2%}",
            'RT P(+)': f"{rt['P(+)']:.1%}",
        })
    print(pd.DataFrame(rows).to_string(index=False))


  Markup Sensitivity — Width=5pct
 Markup Corr Median Corr Sharpe Corr P(+) RT Median RT Mean RT P(+)
   1.05      +8.37%       0.341     58.7%   -14.87% -14.23%    9.4%
   1.10      +7.03%       0.287     56.8%   -12.43% -11.82%   14.7%
   1.15      +5.70%       0.244     55.6%   -10.26%  -9.92%   18.2%

  Markup Sensitivity — Width=10pct
 Markup Corr Median Corr Sharpe Corr P(+) RT Median RT Mean RT P(+)
   1.05     +37.35%       1.125     86.3%    -4.79%  -3.09%   41.0%
   1.10     +30.09%       0.985     83.0%    +0.01%  +1.66%   50.0%
   1.15     +23.60%       0.813     77.6%    +5.34%  +7.32%   59.3%


In [17]:
# ── Viability summary at reference yield ──
print("\n" + "="*60)
print("  VIABILITY MATRIX (reference yield)")
print("="*60)
viab_rows = []
for markup in MARKUPS:
    for wk in WIDTHS:
        res = all_results[(markup, wk, 1.00)]
        df = compute_stats(res)
        lp_med = df.loc[df['Strategy']=='Corridor (two-part)', 'Median Ann'].values[0]
        rt_med = df.loc[df['Strategy']=='RT Pool', 'Median Ann'].values[0]
        viab_rows.append({
            'Markup': markup, 'Width': wk,
            'LP Median': f"{lp_med:+.2%}",
            'RT Median': f"{rt_med:+.2%}",
            'Both Viable': "YES" if (lp_med > 0 and rt_med > 0) else "NO"
        })
print(pd.DataFrame(viab_rows).to_string(index=False))


  VIABILITY MATRIX (reference yield)
 Markup Width LP Median RT Median Both Viable
   1.05  5pct    +8.37%   -14.87%          NO
   1.05 10pct   +37.35%    -4.79%          NO
   1.10  5pct    +7.03%   -12.43%          NO
   1.10 10pct   +30.09%    +0.01%         YES
   1.15  5pct    +5.70%   -10.26%          NO
   1.15 10pct   +23.60%    +5.34%         YES


## Part V: Yield Regime Comparison

How do strategies perform when LP fee yields vary from 0.5x to 1.5x the reference rate?

In [18]:
# ── Yield regime tables ──
# Find optimal markup per width (at reference fee)
optimal_markup = {}
for wk in WIDTHS:
    best_m, best_sharpe = None, -999
    for markup in MARKUPS:
        res = all_results[(markup, wk, 1.00)]
        df = compute_stats(res)
        sh = df.loc[df['Strategy']=='Corridor (two-part)', 'Sharpe'].values[0]
        if sh > best_sharpe:
            best_sharpe = sh
            best_m = markup
    optimal_markup[wk] = best_m
    print(f"Optimal markup for {wk}: {best_m} (Sharpe={best_sharpe:.3f})")

for wk in WIDTHS:
    m = optimal_markup[wk]
    print(f"\n{'='*80}")
    print(f"  Yield Regime Sweep — Width={wk}, Markup={m}")
    print(f"{'='*80}")
    rows = []
    for fm in FEE_MULTS:
        res = all_results[(m, wk, fm)]
        df = compute_stats(res)
        corr = df[df['Strategy'] == 'Corridor (two-part)'].iloc[0]
        rt = df[df['Strategy'] == 'RT Pool'].iloc[0]
        lp = df[df['Strategy'] == 'Plain LP'].iloc[0]
        rows.append({
            'Fee Mult': f"{fm:.2f}",
            'Corr Median': f"{corr['Median Ann']:+.2%}",
            'Corr Sharpe': f"{corr['Sharpe']:.3f}",
            'RT Median': f"{rt['Median Ann']:+.2%}",
            'RT Mean': f"{rt['Mean Ann']:+.2%}",
            'LP Median': f"{lp['Median Ann']:+.2%}",
            'Both OK': "Y" if (corr['Median Ann'] > 0 and rt['Median Ann'] > 0) else "N"
        })
    print(pd.DataFrame(rows).to_string(index=False))

Optimal markup for 5pct: 1.05 (Sharpe=0.341)
Optimal markup for 10pct: 1.05 (Sharpe=1.125)

  Yield Regime Sweep — Width=5pct, Markup=1.05
Fee Mult Corr Median Corr Sharpe RT Median RT Mean LP Median Both OK
    0.50     -47.40%      -2.453   -14.74% -14.28%   -56.49%       N
    0.75     -24.25%      -0.760   -14.65% -14.14%   -37.31%       N
    1.00      +8.37%       0.341   -14.87% -14.23%   -10.09%       N
    1.25     +58.60%       1.080   -14.91% -14.27%   +32.22%       N
    1.50    +126.42%       1.516   -14.63% -14.34%   +88.85%       N

  Yield Regime Sweep — Width=10pct, Markup=1.05
Fee Mult Corr Median Corr Sharpe RT Median RT Mean LP Median Both OK
    0.50     -32.37%      -1.997    -4.66%  -2.71%   -37.06%       N
    0.75      -4.53%      -0.162    -5.48%  -3.25%   -12.32%       N
    1.00     +37.35%       1.125    -4.79%  -3.09%   +27.02%       N
    1.25     +92.43%       1.937    -5.60%  -3.70%   +76.57%       N
    1.50    +171.61%       2.511    -5.46%  -3.96%  +

In [19]:
# ── Heatmap: strategy viability across (markup x yield_regime) ──
fig, axes = plt.subplots(1, len(WIDTHS), figsize=(14, 6))

for wi, wk in enumerate(WIDTHS):
    ax = axes[wi]
    viab = np.zeros((len(MARKUPS), len(FEE_MULTS)))
    for mi, markup in enumerate(MARKUPS):
        for fi, fm in enumerate(FEE_MULTS):
            res = all_results[(markup, wk, fm)]
            df = compute_stats(res)
            lp_ok = df.loc[df['Strategy']=='Corridor (two-part)', 'Median Ann'].values[0] > 0
            rt_ok = df.loc[df['Strategy']=='RT Pool', 'Median Ann'].values[0] > 0
            if lp_ok and rt_ok:
                viab[mi, fi] = 2   # green
            elif lp_ok or rt_ok:
                viab[mi, fi] = 1   # yellow
            else:
                viab[mi, fi] = 0   # red

    from matplotlib.colors import ListedColormap
    cmap = ListedColormap(['#ff4444', '#ffcc00', '#44bb44'])
    im = ax.imshow(viab, cmap=cmap, vmin=0, vmax=2, aspect='auto')
    ax.set_xticks(range(len(FEE_MULTS)))
    ax.set_xticklabels([f"{fm:.2f}x" for fm in FEE_MULTS])
    ax.set_yticks(range(len(MARKUPS)))
    ax.set_yticklabels([f"{m:.2f}" for m in MARKUPS])
    ax.set_xlabel('Fee Multiplier')
    ax.set_ylabel('Markup')
    ax.set_title(f'Viability: {wk}')

    # Annotate cells
    for mi in range(len(MARKUPS)):
        for fi in range(len(FEE_MULTS)):
            v = viab[mi, fi]
            txt = ['Neither', 'One', 'Both'][int(v)]
            ax.text(fi, mi, txt, ha='center', va='center',
                    fontsize=8, fontweight='bold',
                    color='white' if v != 1 else 'black')

plt.suptitle('LP+RT Viability: Green=Both, Yellow=One, Red=Neither',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('data/mc_viability_heatmap.png', bbox_inches='tight')
plt.close()
print("Saved: data/mc_viability_heatmap.png")

Saved: data/mc_viability_heatmap.png


In [20]:
# ── Yield regime line plots: median return vs fee multiplier ──
fig, axes = plt.subplots(1, len(WIDTHS), figsize=(14, 6), sharey=True)
key_strats = ['Plain LP', 'Corridor (two-part)', 'Corridor+Jito', 'RT Pool']
line_colors = {'Plain LP': 'gray', 'Corridor (two-part)': 'blue',
               'Corridor+Jito': 'green', 'RT Pool': 'orange'}

for wi, wk in enumerate(WIDTHS):
    ax = axes[wi]
    m = optimal_markup[wk]
    for strat in key_strats:
        medians = []
        for fm in FEE_MULTS:
            res = all_results[(m, wk, fm)]
            medians.append(np.median(res[strat] - 1))
        ax.plot(FEE_MULTS, medians, 'o-', label=strat,
                color=line_colors[strat], lw=2)
    ax.axhline(0, color='k', ls='--', lw=0.8)
    ax.axhline(BOND_APY, color='red', ls=':', lw=0.8, label='Bond 12%')
    ax.set_xlabel('Fee Multiplier')
    ax.set_ylabel('Median Annual Return' if wi == 0 else '')
    ax.set_title(f'{wk} (markup={m})')
    ax.legend(fontsize=8)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))

plt.suptitle('Median Return vs Fee Regime', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('data/mc_yield_regime_lines.png', bbox_inches='tight')
plt.close()
print("Saved: data/mc_yield_regime_lines.png")

Saved: data/mc_yield_regime_lines.png


In [21]:
# ── Summary: minimum fee mult for viability ──
print("\n" + "="*60)
print("  MINIMUM FEE MULTIPLIER FOR VIABILITY")
print("="*60)
rows = []
for wk in WIDTHS:
    for markup in MARKUPS:
        lp_min = None
        rt_min = None
        for fm in FEE_MULTS:
            res = all_results[(markup, wk, fm)]
            df = compute_stats(res)
            lp_med = df.loc[df['Strategy']=='Corridor (two-part)', 'Median Ann'].values[0]
            rt_med = df.loc[df['Strategy']=='RT Pool', 'Median Ann'].values[0]
            if lp_min is None and lp_med > 0:
                lp_min = fm
            if rt_min is None and rt_med > 0:
                rt_min = fm
        rows.append({
            'Width': wk, 'Markup': markup,
            'LP min fee_mult': f"{lp_min:.2f}" if lp_min else ">1.50",
            'RT min fee_mult': f"{rt_min:.2f}" if rt_min else ">1.50",
        })
print(pd.DataFrame(rows).to_string(index=False))


  MINIMUM FEE MULTIPLIER FOR VIABILITY
Width  Markup LP min fee_mult RT min fee_mult
 5pct    1.05            1.00           >1.50
 5pct    1.10            1.00           >1.50
 5pct    1.15            1.00           >1.50
10pct    1.05            1.00           >1.50
10pct    1.10            1.00            0.50
10pct    1.15            1.00            0.50


## Part VI: Breakeven Fee Rate Sweep

Fine-grained sweep of daily fee rates to find exact breakeven points for each strategy.

In [22]:
# ── Breakeven fee rate sweep ──
FEE_SWEEP = np.linspace(0.001, 0.015, 15)  # 0.1%/day to 1.5%/day
breakeven_results = {}

for markup in MARKUPS:
    for wk, wcfg in WIDTHS.items():
        print(f"Sweeping: markup={markup}, width={wk}...")
        for daily_fee in FEE_SWEEP:
            res = run_block_bootstrap_mc(
                markup, wk, wcfg, weekly_blocks, S0,
                n_paths=N_PATHS_BRK, n_weeks=N_WEEKS,
                daily_fee_override=daily_fee)
            breakeven_results[(markup, wk, daily_fee)] = res

print(f"Breakeven sweep complete: {len(breakeven_results)} configs.")

Sweeping: markup=1.05, width=5pct...


Sweeping: markup=1.05, width=10pct...


Sweeping: markup=1.1, width=5pct...


Sweeping: markup=1.1, width=10pct...


Sweeping: markup=1.15, width=5pct...


Sweeping: markup=1.15, width=10pct...


Breakeven sweep complete: 90 configs.


In [23]:
# ── Find breakeven fee rates ──
print("\n" + "="*70)
print("  BREAKEVEN DAILY FEE RATES (median return = 0)")
print("="*70)

target_strats = ['Plain LP', 'Corridor (two-part)', 'Corridor+Jito',
                 'RT Pool', 'Put Spread', 'Perp Hedge']
rows = []
for markup in MARKUPS:
    for wk in WIDTHS:
        row = {'Markup': markup, 'Width': wk}
        for strat in target_strats:
            medians = []
            for daily_fee in FEE_SWEEP:
                res = breakeven_results[(markup, wk, daily_fee)]
                medians.append(np.median(res[strat] - 1))
            medians = np.array(medians)
            # Find zero crossing
            sign_changes = np.where(np.diff(np.sign(medians)))[0]
            if len(sign_changes) > 0:
                idx = sign_changes[0]
                # Linear interpolation
                f1, f2 = FEE_SWEEP[idx], FEE_SWEEP[idx+1]
                m1, m2 = medians[idx], medians[idx+1]
                be = f1 + (f2 - f1) * (-m1) / (m2 - m1)
                row[strat] = f"{be:.4f}"
            elif medians[0] > 0:
                row[strat] = "<0.001"
            else:
                row[strat] = ">0.015"
        rows.append(row)

print(pd.DataFrame(rows).to_string(index=False))


  BREAKEVEN DAILY FEE RATES (median return = 0)
 Markup Width Plain LP Corridor (two-part) Corridor+Jito RT Pool Put Spread Perp Hedge
   1.05  5pct   0.0070              0.0061        0.0060  >0.015     0.0050     0.0080
   1.05 10pct   0.0036              0.0034        0.0033  >0.015     0.0011     0.0046
   1.10  5pct   0.0070              0.0062        0.0061  >0.015     0.0051     0.0081
   1.10 10pct   0.0037              0.0036        0.0035  0.0026     0.0011     0.0045
   1.15  5pct   0.0068              0.0062        0.0061  >0.015     0.0051     0.0081
   1.15 10pct   0.0038              0.0038        0.0037  <0.001     0.0011     0.0046


In [24]:
# ── Breakeven plots: median return vs fee rate ──
fig, axes = plt.subplots(len(MARKUPS), len(WIDTHS), figsize=(16, 14), sharey=True)
plot_strats = ['Plain LP', 'Corridor (two-part)', 'Corridor+Jito',
               'RT Pool', 'Put Spread', 'Perp Hedge']
strat_colors = {'Plain LP': 'gray', 'Corridor (two-part)': 'blue',
                'Corridor+Jito': 'green', 'RT Pool': 'orange',
                'Put Spread': 'purple', 'Perp Hedge': 'red'}

for i, markup in enumerate(MARKUPS):
    for j, wk in enumerate(WIDTHS):
        ax = axes[i, j]
        for strat in plot_strats:
            medians = [np.median(breakeven_results[(markup, wk, f)][strat] - 1)
                       for f in FEE_SWEEP]
            ax.plot(FEE_SWEEP * 100, medians, 'o-', ms=3, lw=1.5,
                    label=strat, color=strat_colors[strat])
        ax.axhline(0, color='k', ls='--', lw=0.8)
        ax.axhline(BOND_APY, color='darkgreen', ls=':', lw=0.8, alpha=0.5)
        # Mark reference fee
        ref_fee = WIDTHS[wk]['daily_fee']
        ax.axvline(ref_fee * 100, color='gray', ls=':', lw=1, alpha=0.7)
        ax.set_title(f"Markup={markup}, {wk}")
        ax.set_xlabel('Daily Fee Rate (%)')
        ax.set_ylabel('Median Ann. Return' if j == 0 else '')
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
        if i == 0 and j == 1:
            ax.legend(fontsize=7, loc='upper left')

plt.suptitle('Breakeven Fee Rate Sweep', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('data/mc_breakeven_fee_rates.png', bbox_inches='tight')
plt.close()
print("Saved: data/mc_breakeven_fee_rates.png")

Saved: data/mc_breakeven_fee_rates.png


In [25]:
# ── Breakeven key findings ──
print("\nKey findings from breakeven sweep:")
for wk in WIDTHS:
    ref = WIDTHS[wk]['daily_fee']
    print(f"\n  {wk} (reference fee = {ref:.4f}/day = {ref*100:.2f}%/day):")
    for markup in MARKUPS:
        res_ref = all_results[(markup, wk, 1.00)]
        df = compute_stats(res_ref)
        corr_med = df.loc[df['Strategy']=='Corridor (two-part)', 'Median Ann'].values[0]
        rt_med = df.loc[df['Strategy']=='RT Pool', 'Median Ann'].values[0]
        print(f"    markup={markup}: Corridor={corr_med:+.2%}, RT={rt_med:+.2%}")


Key findings from breakeven sweep:

  5pct (reference fee = 0.0065/day = 0.65%/day):
    markup=1.05: Corridor=+8.37%, RT=-14.87%
    markup=1.1: Corridor=+7.03%, RT=-12.43%
    markup=1.15: Corridor=+5.70%, RT=-10.26%

  10pct (reference fee = 0.0045/day = 0.45%/day):
    markup=1.05: Corridor=+37.35%, RT=-4.79%
    markup=1.1: Corridor=+30.09%, RT=+0.01%
    markup=1.15: Corridor=+23.60%, RT=+5.34%


## Part VII: Head-to-Head Comparison & RT Deep Dive

Win rates across all yield regimes. RT decomposition: premiums, claims, loss ratio.

In [26]:
# ── Corridor vs each alternative: win count across all yield regimes ──
competitors = ['Plain LP', 'Put Spread', 'Perp Hedge', 'Bond']
print("\n" + "="*70)
print("  HEAD-TO-HEAD: Corridor (two-part) vs Alternatives")
print("  Win = higher median annual return across all (markup, width, fee_mult)")
print("="*70)

wins = {c: 0 for c in competitors}
total = 0
for markup in MARKUPS:
    for wk in WIDTHS:
        for fm in FEE_MULTS:
            res = all_results[(markup, wk, fm)]
            corr_med = np.median(res['Corridor (two-part)'] - 1)
            for comp in competitors:
                if comp == 'Bond':
                    comp_med = BOND_APY
                else:
                    comp_med = np.median(res[comp] - 1)
                if corr_med > comp_med:
                    wins[comp] += 1
            total += 1

print(f"\nTotal scenarios: {total}")
for comp in competitors:
    print(f"  Corridor beats {comp:15s}: {wins[comp]:2d}/{total} "
          f"({wins[comp]/total:.0%})")


  HEAD-TO-HEAD: Corridor (two-part) vs Alternatives
  Win = higher median annual return across all (markup, width, fee_mult)

Total scenarios: 30
  Corridor beats Plain LP       : 25/30 (83%)
  Corridor beats Put Spread     :  0/30 (0%)
  Corridor beats Perp Hedge     : 30/30 (100%)
  Corridor beats Bond           : 15/30 (50%)


In [27]:
# ── RT decomposition: detailed analysis at optimal markup ──
def rt_decomposition(markup, wk, wcfg, weekly_blocks, S0, fm=1.00):
    """Run MC and track RT premiums/claims per week."""
    w = wcfg['w']
    daily_fee = wcfg['daily_fee'] * fm
    fs_bps = wcfg['fs_bps']
    ann_vol = log_returns.std() * np.sqrt(365)
    n_paths = 2000
    n_weeks = N_WEEKS

    S_median = np.median(weekly_blocks[:, 0])
    pos_med = make_position(S_median, w)
    fv_med = fv_quadrature(S_median, ann_vol, T=T_WEEK, B=pos_med['B'],
                           Cap=pos_med['Cap'], L=N_LIQUIDITY,
                           p_l=pos_med['p_l'], p_u=pos_med['p_u'])
    V0_median = pos_med['V0']

    n_blocks = len(weekly_blocks)
    block_indices = rng.integers(0, n_blocks, size=(n_paths, n_weeks))

    total_prem = np.zeros(n_paths)
    total_claims = np.zeros(n_paths)
    weeks_with_claims = np.zeros(n_paths)

    for week_idx in range(n_weeks):
        bidx = block_indices[:, week_idx]
        blocks = weekly_blocks[bidx]
        S_start = blocks[:, 0]
        S_end = blocks[:, 7]
        daily_prices = blocks[:, 1:7]

        p_l = S_start * (1 - w)
        p_u = S_start * (1 + w)
        V0 = cl_value_vec(S_start, N_LIQUIDITY, p_l, p_u)
        Cap_arr = V0 * 0.30
        fv_scaled = fv_med * (V0 / V0_median)

        # Fees and vol indicator
        fees = daily_fee * V0 * 7
        in_range_days = np.sum(
            (daily_prices >= p_l[:, None]) & (daily_prices <= p_u[:, None]), axis=1)
        in_range_entry = ((S_start >= p_l) & (S_start <= p_u)).astype(float)
        in_range_exit  = ((S_end >= p_l) & (S_end <= p_u)).astype(float)
        in_range_frac = (in_range_days + in_range_entry + in_range_exit) / 8.0
        fees = fees * in_range_frac

        daily_log_rets = np.diff(np.log(blocks), axis=1)
        block_vol = np.std(daily_log_rets, axis=1) * np.sqrt(365)
        vol_ind = np.clip(block_vol / ann_vol, 0.5, 2.0)

        weekly_fee_exp = V0 * daily_fee * 7
        beta = (markup - ALPHA) * fv_scaled / np.maximum(weekly_fee_exp, 1.0)
        prem_twopart = ALPHA * fv_scaled * vol_ind + beta * fees

        payout = corridor_payoff_vec(S_end, S_start, p_l, Cap_arr,
                                     N_LIQUIDITY, p_l, p_u)

        n_certs = np.maximum(1, np.floor(V0 * 0.30 / Cap_arr)).astype(int)
        total_prem += n_certs * prem_twopart * 0.985
        total_claims += n_certs * payout
        weeks_with_claims += (payout > 0).astype(float)

    return {
        'total_prem': total_prem,
        'total_claims': total_claims,
        'weeks_with_claims': weeks_with_claims,
        'loss_ratio': total_claims / np.maximum(total_prem, 1e-10),
    }

for wk in WIDTHS:
    m = optimal_markup[wk]
    print(f"\n{'='*60}")
    print(f"  RT Decomposition: width={wk}, markup={m}")
    print(f"{'='*60}")
    decomp = rt_decomposition(m, wk, WIDTHS[wk], weekly_blocks, S0)
    print(f"  Premiums collected (median): ${np.median(decomp['total_prem']):.2f}")
    print(f"  Claims paid (median):        ${np.median(decomp['total_claims']):.2f}")
    print(f"  Loss ratio (median):         {np.median(decomp['loss_ratio']):.2%}")
    print(f"  Loss ratio (mean):           {np.mean(decomp['loss_ratio']):.2%}")
    print(f"  Weeks with claims (median):  {np.median(decomp['weeks_with_claims']):.0f}/{N_WEEKS}")
    print(f"  P(loss ratio > 1):           {np.mean(decomp['loss_ratio'] > 1):.1%}")


  RT Decomposition: width=5pct, markup=1.05
  Premiums collected (median): $3603.39
  Claims paid (median):        $4616.44
  Loss ratio (median):         128.11%
  Loss ratio (mean):           128.82%
  Weeks with claims (median):  27/52
  P(loss ratio > 1):           91.8%

  RT Decomposition: width=10pct, markup=1.05
  Premiums collected (median): $12849.17
  Claims paid (median):        $13520.53
  Loss ratio (median):         105.37%
  Loss ratio (mean):           106.01%
  Weeks with claims (median):  27/52
  P(loss ratio > 1):           59.8%


In [28]:
# ── RT wealth path percentile bands ──
def rt_wealth_paths(markup, wk, wcfg, weekly_blocks, S0, n_paths=2000):
    """Track RT wealth week by week."""
    w = wcfg['w']
    daily_fee = wcfg['daily_fee']
    ann_vol = log_returns.std() * np.sqrt(365)

    S_median = np.median(weekly_blocks[:, 0])
    pos_med = make_position(S_median, w)
    fv_med = fv_quadrature(S_median, ann_vol, T=T_WEEK, B=pos_med['B'],
                           Cap=pos_med['Cap'], L=N_LIQUIDITY,
                           p_l=pos_med['p_l'], p_u=pos_med['p_u'])
    V0_median = pos_med['V0']

    n_blocks = len(weekly_blocks)
    block_indices = rng.integers(0, n_blocks, size=(n_paths, N_WEEKS))

    wealth_track = np.ones((n_paths, N_WEEKS + 1))

    for week_idx in range(N_WEEKS):
        bidx = block_indices[:, week_idx]
        blocks = weekly_blocks[bidx]
        S_start, S_end = blocks[:, 0], blocks[:, 7]
        daily_prices = blocks[:, 1:7]
        p_l = S_start * (1 - w)
        p_u = S_start * (1 + w)
        V0 = cl_value_vec(S_start, N_LIQUIDITY, p_l, p_u)
        Cap_arr = V0 * 0.30
        fv_scaled = fv_med * (V0 / V0_median)

        fees = daily_fee * V0 * 7
        in_range_days = np.sum(
            (daily_prices >= p_l[:, None]) & (daily_prices <= p_u[:, None]), axis=1)
        in_range_entry = ((S_start >= p_l) & (S_start <= p_u)).astype(float)
        in_range_exit  = ((S_end >= p_l) & (S_end <= p_u)).astype(float)
        fees = fees * (in_range_days + in_range_entry + in_range_exit) / 8.0

        daily_log_rets = np.diff(np.log(blocks), axis=1)
        block_vol = np.std(daily_log_rets, axis=1) * np.sqrt(365)
        vol_ind = np.clip(block_vol / ann_vol, 0.5, 2.0)

        weekly_fee_exp = V0 * daily_fee * 7
        beta = (markup - ALPHA) * fv_scaled / np.maximum(weekly_fee_exp, 1.0)
        prem = ALPHA * fv_scaled * vol_ind + beta * fees

        payout = corridor_payoff_vec(S_end, S_start, p_l, Cap_arr,
                                     N_LIQUIDITY, p_l, p_u)

        n_certs = np.maximum(1, np.floor(V0 * 0.30 / Cap_arr)).astype(int)
        ret_rt = (n_certs * prem * 0.985 - n_certs * payout) / V0
        wealth_track[:, week_idx + 1] = wealth_track[:, week_idx] * (1 + ret_rt)

    return wealth_track

fig, axes = plt.subplots(1, len(WIDTHS), figsize=(14, 5), sharey=True)
for wi, wk in enumerate(WIDTHS):
    ax = axes[wi]
    m = optimal_markup[wk]
    wt = rt_wealth_paths(m, wk, WIDTHS[wk], weekly_blocks, S0)
    weeks = np.arange(N_WEEKS + 1)

    p10 = np.percentile(wt, 10, axis=0)
    p25 = np.percentile(wt, 25, axis=0)
    p50 = np.median(wt, axis=0)
    p75 = np.percentile(wt, 75, axis=0)
    p90 = np.percentile(wt, 90, axis=0)

    ax.fill_between(weeks, p10, p90, alpha=0.15, color='orange', label='10-90th')
    ax.fill_between(weeks, p25, p75, alpha=0.3, color='orange', label='25-75th')
    ax.plot(weeks, p50, 'o-', ms=2, color='darkorange', lw=2, label='Median')
    ax.axhline(1.0, color='k', ls='--', lw=0.8)
    ax.set_title(f'RT Wealth: {wk} (markup={m})')
    ax.set_xlabel('Week')
    ax.set_ylabel('Wealth Multiple' if wi == 0 else '')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('data/mc_rt_wealth_paths.png', bbox_inches='tight')
plt.close()
print("Saved: data/mc_rt_wealth_paths.png")

Saved: data/mc_rt_wealth_paths.png


In [29]:
# ── Corridor (two-part) wealth path bands for comparison ──
def corr_wealth_paths(markup, wk, wcfg, weekly_blocks, S0, n_paths=2000):
    w = wcfg['w']
    daily_fee = wcfg['daily_fee']
    ann_vol = log_returns.std() * np.sqrt(365)

    S_median = np.median(weekly_blocks[:, 0])
    pos_med = make_position(S_median, w)
    fv_med = fv_quadrature(S_median, ann_vol, T=T_WEEK, B=pos_med['B'],
                           Cap=pos_med['Cap'], L=N_LIQUIDITY,
                           p_l=pos_med['p_l'], p_u=pos_med['p_u'])
    V0_median = pos_med['V0']

    n_blocks = len(weekly_blocks)
    block_indices = rng.integers(0, n_blocks, size=(n_paths, N_WEEKS))
    wealth_track = np.ones((n_paths, N_WEEKS + 1))

    for week_idx in range(N_WEEKS):
        bidx = block_indices[:, week_idx]
        blocks = weekly_blocks[bidx]
        S_start, S_end = blocks[:, 0], blocks[:, 7]
        daily_prices = blocks[:, 1:7]
        p_l = S_start * (1 - w)
        p_u = S_start * (1 + w)
        V0 = cl_value_vec(S_start, N_LIQUIDITY, p_l, p_u)
        V1 = cl_value_vec(S_end, N_LIQUIDITY, p_l, p_u)
        IL = V1 - V0
        Cap_arr = V0 * 0.30
        fv_scaled = fv_med * (V0 / V0_median)

        fees = daily_fee * V0 * 7
        in_range_days = np.sum(
            (daily_prices >= p_l[:, None]) & (daily_prices <= p_u[:, None]), axis=1)
        in_range_entry = ((S_start >= p_l) & (S_start <= p_u)).astype(float)
        in_range_exit  = ((S_end >= p_l) & (S_end <= p_u)).astype(float)
        fees = fees * (in_range_days + in_range_entry + in_range_exit) / 8.0

        daily_log_rets = np.diff(np.log(blocks), axis=1)
        block_vol = np.std(daily_log_rets, axis=1) * np.sqrt(365)
        vol_ind = np.clip(block_vol / ann_vol, 0.5, 2.0)

        weekly_fee_exp = V0 * daily_fee * 7
        beta = (markup - ALPHA) * fv_scaled / np.maximum(weekly_fee_exp, 1.0)
        prem = ALPHA * fv_scaled * vol_ind + beta * fees

        payout = corridor_payoff_vec(S_end, S_start, p_l, Cap_arr,
                                     N_LIQUIDITY, p_l, p_u)

        ret = (IL + fees + payout - prem) / V0
        wealth_track[:, week_idx + 1] = wealth_track[:, week_idx] * (1 + ret)

    return wealth_track

fig, axes = plt.subplots(1, len(WIDTHS), figsize=(14, 5), sharey=True)
for wi, wk in enumerate(WIDTHS):
    ax = axes[wi]
    m = optimal_markup[wk]
    wt = corr_wealth_paths(m, wk, WIDTHS[wk], weekly_blocks, S0)
    weeks = np.arange(N_WEEKS + 1)

    p10 = np.percentile(wt, 10, axis=0)
    p25 = np.percentile(wt, 25, axis=0)
    p50 = np.median(wt, axis=0)
    p75 = np.percentile(wt, 75, axis=0)
    p90 = np.percentile(wt, 90, axis=0)

    ax.fill_between(weeks, p10, p90, alpha=0.15, color='blue', label='10-90th')
    ax.fill_between(weeks, p25, p75, alpha=0.3, color='blue', label='25-75th')
    ax.plot(weeks, p50, 'o-', ms=2, color='darkblue', lw=2, label='Median')
    ax.axhline(1.0, color='k', ls='--', lw=0.8)

    # Overlay bond
    bond_path = [(1 + BOND_APY)**(i/52) for i in range(N_WEEKS+1)]
    ax.plot(weeks, bond_path, 'g--', lw=1.5, label='Bond')

    ax.set_title(f'Corridor Wealth: {wk} (markup={m})')
    ax.set_xlabel('Week')
    ax.set_ylabel('Wealth Multiple' if wi == 0 else '')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('data/mc_corridor_wealth_paths.png', bbox_inches='tight')
plt.close()
print("Saved: data/mc_corridor_wealth_paths.png")

Saved: data/mc_corridor_wealth_paths.png


## Part VIII: Conclusions

Summary recommendations by configuration and final viability verdict.

In [30]:
# ── Summary: recommended markup per width per yield regime ──
print("\n" + "="*70)
print("  RECOMMENDED MARKUP PER CONFIGURATION")
print("  (maximizes Corridor Sharpe while keeping RT median > 0)")
print("="*70)

rows = []
for wk in WIDTHS:
    for fm in FEE_MULTS:
        best_m, best_sh = None, -999
        for markup in MARKUPS:
            res = all_results[(markup, wk, fm)]
            df = compute_stats(res)
            corr_sh = df.loc[df['Strategy']=='Corridor (two-part)', 'Sharpe'].values[0]
            rt_med = df.loc[df['Strategy']=='RT Pool', 'Median Ann'].values[0]
            if rt_med > 0 and corr_sh > best_sh:
                best_sh = corr_sh
                best_m = markup
        if best_m is None:
            # Fallback: just max Sharpe without RT constraint
            for markup in MARKUPS:
                res = all_results[(markup, wk, fm)]
                df = compute_stats(res)
                corr_sh = df.loc[df['Strategy']=='Corridor (two-part)', 'Sharpe'].values[0]
                if corr_sh > best_sh:
                    best_sh = corr_sh
                    best_m = markup

        res = all_results[(best_m, wk, fm)]
        df = compute_stats(res)
        corr_med = df.loc[df['Strategy']=='Corridor (two-part)', 'Median Ann'].values[0]
        rt_med = df.loc[df['Strategy']=='RT Pool', 'Median Ann'].values[0]

        rows.append({
            'Width': wk, 'Fee Mult': f"{fm:.2f}",
            'Best Markup': best_m if best_m else 'N/A',
            'Corr Median': f"{corr_med:+.2%}",
            'Corr Sharpe': f"{best_sh:.3f}",
            'RT Median': f"{rt_med:+.2%}",
            'Viable': "YES" if (corr_med > 0 and rt_med > 0) else "NO"
        })

print(pd.DataFrame(rows).to_string(index=False))


  RECOMMENDED MARKUP PER CONFIGURATION
  (maximizes Corridor Sharpe while keeping RT median > 0)
Width Fee Mult  Best Markup Corr Median Corr Sharpe RT Median Viable
 5pct     0.50         1.05     -47.40%      -2.453   -14.74%     NO
 5pct     0.75         1.05     -24.25%      -0.760   -14.65%     NO
 5pct     1.00         1.05      +8.37%       0.341   -14.87%     NO
 5pct     1.25         1.05     +58.60%       1.080   -14.91%     NO
 5pct     1.50         1.05    +126.42%       1.516   -14.63%     NO
10pct     0.50         1.10     -36.24%      -2.383    +0.09%     NO
10pct     0.75         1.15     -13.23%      -0.602    +6.15%     NO
10pct     1.00         1.10     +30.09%       0.985    +0.01%    YES
10pct     1.25         1.15     +74.92%       1.746    +5.16%    YES
10pct     1.50         1.15    +145.20%       2.382    +4.91%    YES


In [31]:
# ── Final verdict ──
print("\n" + "="*70)
print("  FINAL VERDICT: PRODUCT VIABILITY")
print("="*70)

# Count viable configs
n_viable = 0
n_total = 0
for markup in MARKUPS:
    for wk in WIDTHS:
        for fm in FEE_MULTS:
            res = all_results[(markup, wk, fm)]
            df = compute_stats(res)
            lp_med = df.loc[df['Strategy']=='Corridor (two-part)', 'Median Ann'].values[0]
            rt_med = df.loc[df['Strategy']=='RT Pool', 'Median Ann'].values[0]
            n_total += 1
            if lp_med > 0 and rt_med > 0:
                n_viable += 1

print(f"\n  Configurations tested: {n_total}")
print(f"  Both LP+RT viable:    {n_viable} ({n_viable/n_total:.0%})")
print(f"  Method: block bootstrap ({weekly_blocks.shape[0]} blocks of 8 days)")
print(f"  Paths: {N_PATHS_MAIN} (main), {N_PATHS_BRK} (breakeven)")
print(f"  Historical data: {len(hist_prices)} daily closes")
print(f"  Current SOL/USDC: ${S0:.2f}")
print(f"  Annualized vol: {log_returns.std()*np.sqrt(365):.1%}")

# Best configs
print(f"\n  Best configurations (both sides viable):")
best_configs = []
for markup in MARKUPS:
    for wk in WIDTHS:
        for fm in FEE_MULTS:
            res = all_results[(markup, wk, fm)]
            df = compute_stats(res)
            lp_med = df.loc[df['Strategy']=='Corridor (two-part)', 'Median Ann'].values[0]
            rt_med = df.loc[df['Strategy']=='RT Pool', 'Median Ann'].values[0]
            lp_sh = df.loc[df['Strategy']=='Corridor (two-part)', 'Sharpe'].values[0]
            if lp_med > 0 and rt_med > 0:
                best_configs.append((lp_sh, markup, wk, fm, lp_med, rt_med))

best_configs.sort(reverse=True)
for sh, m, wk, fm, lp, rt in best_configs[:5]:
    print(f"    markup={m}, {wk}, fee={fm:.2f}x -> "
          f"Corr={lp:+.2%} (Sharpe={sh:.3f}), RT={rt:+.2%}")

if not best_configs:
    print("    No configurations found where both LP and RT are viable.")
    print("    The corridor product may need parameter adjustment.")

print(f"\n  Block bootstrap preserves empirical fat tails (kurtosis="
      f"{stats.kurtosis(log_returns):.2f}) and volatility clustering.")
print(f"  Results are more conservative than naive IID bootstrap.")
print("\n  === Simulation complete ===")


  FINAL VERDICT: PRODUCT VIABILITY



  Configurations tested: 30
  Both LP+RT viable:    4 (13%)
  Method: block bootstrap (358 blocks of 8 days)
  Paths: 5000 (main), 2000 (breakeven)
  Historical data: 365 daily closes
  Current SOL/USDC: $82.30
  Annualized vol: 73.5%

  Best configurations (both sides viable):
    markup=1.15, 10pct, fee=1.50x -> Corr=+145.20% (Sharpe=2.382), RT=+4.91%
    markup=1.15, 10pct, fee=1.25x -> Corr=+74.92% (Sharpe=1.746), RT=+5.16%
    markup=1.1, 10pct, fee=1.00x -> Corr=+30.09% (Sharpe=0.985), RT=+0.01%
    markup=1.15, 10pct, fee=1.00x -> Corr=+23.60% (Sharpe=0.813), RT=+5.34%

  Block bootstrap preserves empirical fat tails (kurtosis=1.63) and volatility clustering.
  Results are more conservative than naive IID bootstrap.

  === Simulation complete ===
